# SE3a_model_setup

**Author:** MZH Taki, University of Oulu  

---

## Purpose
This notebook is **Step 3a** of the pipeline — run **once** to set up and calibrate the model.

| Step | What | When |
|------|------|------|
| Cell 4 | Base model run (uncalibrated baseline) | Once |
| Cell 5–6 | Sensitivity analysis — Sobol indices for 63 parameters | Once |
| Cell 7 | Calibration — NSGA2 evolutionary optimisation | Once (~20–25 hrs) |
| Cell 8–9 | Calibrated validation run + performance metrics | Once |

All outputs are written to disk. **SE3b reads from these outputs daily** without re-running anything here.

## Reproducibility guarantee
- Fixed random seeds are set for both SALib (Sobol sampling) and pymoo (NSGA2) so any user running this notebook in the same Docker environment will get the same results.
- SWAT+ executable version: **61.0.2** (Linux x86_64)
- Python environment versions are printed in Cell 1 and saved to `model/config/environment.txt`

## Input data
- `TxtInOut_test/` — from Zenodo via SE1
- `Observed_data_standard_format_1.txt` — HERTTA station 7300100 (Oulankajoki), formatted in SE3a Cell 3

## Output files written by this notebook

| File | Location | Description |
|------|----------|-------------|
| `environment.txt` | `model/config/` | Package versions for reproducibility |
| `model_config.yaml` | `model/config/` | All settings, periods, algorithm config |
| `Observed_data_standard_format_1.txt` | `processeddata/` | Formatted HERTTA observed flow |
| `sensitivity_results.csv` | `predictions/` | Sobol S1 and ST indices for all 63 parameters |
| `optimized_parameters.csv` | `model/` | Best NSGA2 parameter set |
| `evaluation_metrics.csv` | `predictions/` | NSE, KGE, R², PBIAS for calibration and validation |
| `predictions_calval.csv` | `predictions/` | Daily observed and simulated flow for plotting |
| `hydrograph_calval.png` | `predictions/` | Calibration and validation hydrograph |

## Cell 1 — Load dependencies and print environment versions

Versions are printed and saved so anyone reproducing this work can verify their environment matches.

In [1]:
# Modify cell
from pathlib import Path
import os
import glob
import shutil
import datetime
import time
import re
import yaml
import logging
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pySWATPlus

# Print all package versions for reproducibility
import importlib
packages = ['pySWATPlus', 'pandas', 'numpy', 'matplotlib', 'yaml', 'SALib', 'pymoo']
version_lines = []
for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, '__version__', 'unknown')
    except ImportError:
        ver = 'not installed'
    line = f"{pkg}=={ver}"
    print(line)
    version_lines.append(line)

import sys
print(f"python=={sys.version.split()[0]}")
print(f"swatplus_executable==61.0.2")

# Logging setup — shows calibration and sensitivity progress in the notebook
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

print("\nAll dependencies loaded.")

pySWATPlus==1.3.0
pandas==3.0.0
numpy==2.3.5
matplotlib==3.10.8
yaml==6.0.1
SALib==unknown
pymoo==0.6.1.6
python==3.11.6
swatplus_executable==61.0.2

All dependencies loaded.


## Cell 2 — Directory setup and save environment + config

All outputs go outside the repository. The environment versions and model config are saved
immediately so the record exists even if later cells fail.

In [2]:
# Automatically generated cell — resolve external directories
notebook_dir  = Path.cwd()
project_root  = notebook_dir.parent
external_base = project_root.parent

proc_data_dir = external_base / "oulanka_swatplus_processeddata"
model_dir     = proc_data_dir / "model"
config_dir    = model_dir / "config"
pred_dir      = proc_data_dir / "predictions"

for d in [model_dir, config_dir, pred_dir]:
    d.mkdir(parents=True, exist_ok=True)

# ── USER SETTINGS ─────────────────────────────────────────────────────────────
BASE_DIR     = str(external_base / "oulanka_swatplus_rawdata" / "zenodo_data")  # <-- update if needed
TXTINOUT_DIR = os.path.join(BASE_DIR, "TxtInOut_test")   # base model folder from Zenodo
TARGET_UNIT  = 891         # SWAT+ channel unit ID for the Oulanka main outlet
CHANNEL_NAME = 'cha1134'   # channel name in SWAT+ output files  <-- update if needed
BASE_YEAR    = 1990        # first year of SWAT+ climate data

# Observed streamflow file (produced in Cell 3)
OBSERVED_FILE = str(proc_data_dir / "Observed_data_standard_format_1.txt")

# Random seeds — fix these to guarantee identical results on any machine
SOBOL_SEED = 42    # passed to SALib Sobol sampler
NSGA2_SEED = 42    # passed to pymoo NSGA2 algorithm

# Save environment versions to disk
env_path = config_dir / "environment.txt"
with open(env_path, 'w') as f:
    f.write("# Package versions for reproducibility\n")
    for line in version_lines:
        f.write(line + "\n")
    f.write(f"python=={sys.version.split()[0]}\n")
    f.write("swatplus_executable==61.0.2\n")
print(f"Environment versions saved: {env_path}")

# Save model configuration to YAML
model_config = {
    "model"            : "SWAT+ v61.0.2",
    "catchment"        : "Oulanka River, Finland",
    "outlet_unit"      : TARGET_UNIT,
    "channel_name"     : CHANNEL_NAME,
    "obs_station_id"   : "7300100",
    "base_year"        : BASE_YEAR,
    "sobol_seed"       : SOBOL_SEED,
    "nsga2_seed"       : NSGA2_SEED,
    "objective"        : "NSE",
    "algorithm"        : "NSGA2",
    "n_gen"            : 100,
    "pop_size"         : 120,
    "periods": {
        "simulation_start"  : "01-Jan-1997",
        "simulation_end"    : "31-Dec-2022",
        "warmup_years"      : 2,
        "sensitivity_start" : "01-Jan-1997",
        "sensitivity_end"   : "31-Dec-2002",
        "calibration_start" : "01-Jan-1999",
        "calibration_end"   : "31-Dec-2006",
        "validation_start"  : "01-Jan-2007",
        "validation_end"    : "31-Dec-2010"
    }
}

config_path = config_dir / "model_config.yaml"
with open(config_path, 'w') as f:
    yaml.dump(model_config, f, default_flow_style=False)

print(f"Model config saved      : {config_path}")
print(f"Processed data directory: {proc_data_dir.resolve()}")
print(f"Predictions directory   : {pred_dir.resolve()}")

Environment versions saved: /home/jovyan/oulanka_swatplus_processeddata/model/config/environment.txt
Model config saved      : /home/jovyan/oulanka_swatplus_processeddata/model/config/model_config.yaml
Processed data directory: /home/jovyan/oulanka_swatplus_processeddata
Predictions directory   : /home/jovyan/oulanka_swatplus_processeddata/predictions


## Cell 3 — Convert HERTTA observed flow to standard format

Observed streamflow is downloaded manually from the Finnish Environment Institute HERTTA database  
(https://wwwp2.ymparisto.fi/scripts/kirjaudu.asp — account required).  
Station 7300100 (Oulankajoki) provides daily mean discharge in m³/s.

The converter below parses the raw HERTTA export format and saves a standard CSV
readable by `pySWATPlus` calibration and sensitivity interfaces.

In [3]:
# ==========================================
# CELL 3: HERTTA FORMAT CONVERTER
# ==========================================

def convert_hertta(input_file, output_file):
    """
    Convert HERTTA database export to standard pySWATPlus-compatible CSV.

    HERTTA input format:
        Selected observation stations
        "7300100","Oulankajoki","Virtaama"
        "Date","7300100_Q","Flag"
        01.01.1966,8.10,=

    Standard output format:
        date,7300100_q
        01-01-1966,8.1
    """
    with open(input_file, 'r', encoding='utf-8-sig') as f:
        lines = f.readlines()

    station_id, data_start = None, None
    for i, line in enumerate(lines):
        stripped = line.strip()
        if station_id is None:
            match = re.match(r'^"(\d{5,10})","(.+?)","(.+?)"', stripped)
            if match:
                station_id   = match.group(1)
                station_name = match.group(2)
                continue
        if re.match(r'^"?Date"?,', stripped, re.IGNORECASE):
            data_start = i + 1
            break

    if data_start is None:
        raise ValueError("Could not find 'Date' header in HERTTA file")

    print(f"  Station : {station_id} — {station_name}")

    df = pd.read_csv(
        input_file, skiprows=data_start, header=None,
        names=['date', 'value', 'flag'],
        usecols=[0, 1], na_values=['', ' ', 'NA'], encoding='utf-8-sig'
    )
    df['date'] = pd.to_datetime(df['date'], format='%d.%m.%Y')
    df = df.dropna(subset=['value']).sort_values('date').reset_index(drop=True)

    col_name = f"{station_id}_q"
    result   = pd.DataFrame({
        'date'   : df['date'].dt.strftime('%d-%m-%Y'),
        col_name : df['value']
    })
    result.to_csv(output_file, index=False)

    print(f"  Rows    : {len(result)}")
    print(f"  Period  : {result['date'].iloc[0]} to {result['date'].iloc[-1]}")
    print(f"  Saved   : {output_file}")
    return result


# Update input_file path to point to your HERTTA download on this machine
hertta_raw = "/home/jovyan/Taki_Thesis/Observed/Hertta_data/Oulanka_Observed-Hertta.txt"  # <-- update

df_obs_flow = convert_hertta(
    input_file  = hertta_raw,
    output_file = OBSERVED_FILE
)

  Station : 7300100 — Oulankajoki
  Rows    : 21944
  Period  : 01-01-1966 to 08-02-2026
  Saved   : /home/jovyan/oulanka_swatplus_processeddata/Observed_data_standard_format_1.txt


## Cell 4 — Base model run (uncalibrated baseline)

Run the model with default parameters to establish a baseline NSE.  
This is used in SE4 to show how much calibration improved model performance.

In [4]:
# ==========================================
# CELL 4: BASE MODEL RUN (uncalibrated)
# ==========================================

base_sim_dir = os.path.join(BASE_DIR, "model_run_base")
if not os.path.exists(base_sim_dir):
    os.makedirs(base_sim_dir)

# Initialize and copy required SWAT+ files
base_reader = pySWATPlus.TxtinoutReader(tio_dir=TXTINOUT_DIR)
base_reader.copy_required_files(sim_dir=base_sim_dir)
base_sim_reader = pySWATPlus.TxtinoutReader(tio_dir=base_sim_dir)

# Configure and run using the new pySWATPlus API
# (begin_date, end_date, warmup passed directly to run_swat)
base_sim_reader.disable_csv_print()
base_sim_reader.enable_object_in_print_prt(
    obj='channel_sd', daily=True, monthly=True, yearly=True, avann=True
)

print("Starting base model run (uncalibrated)...")
t0 = time.time()

base_sim_reader.run_swat(
    begin_date = model_config['periods']['simulation_start'],
    end_date   = model_config['periods']['simulation_end'],
    warmup     = model_config['periods']['warmup_years']
)

elapsed = time.time() - t0
mins, secs = divmod(elapsed, 60)
print(f"Base run complete. Runtime: {int(mins)} min {secs:.1f} s")
print(f"Output directory: {base_sim_dir}")

TypeError: Expected exactly one executable file in the parent folder, but found none or multiple

## Cell 5 — Sensitivity analysis configuration

Configure the simulation environment for the Sobol sensitivity analysis.
A shorter period (1997–2002) is used here to keep compute time manageable.

> **Warning (from pySWATPlus docs):** After running this configuration cell,
> do **not** re-run it before running Cell 6. Re-running may overwrite configuration
> files and affect sensitivity results.

In [ ]:
# ==========================================
# CELL 5: SENSITIVITY ANALYSIS — CONFIGURATION
# Run this cell ONCE before Cell 6. Do not re-run.
# ==========================================

sens_sim_dir     = os.path.join(BASE_DIR, "sensitivity", "Model_Run_Sensitivity")
sens_results_dir = os.path.join(BASE_DIR, "sensitivity", "Sensitivity_Results")

for folder in [sens_sim_dir, sens_results_dir]:
    if os.path.exists(folder):
        shutil.rmtree(folder)
    os.makedirs(folder)

# Copy SWAT+ files and configure for sensitivity period
sens_base_reader = pySWATPlus.TxtinoutReader(tio_dir=TXTINOUT_DIR)
sens_base_reader.copy_required_files(sim_dir=sens_sim_dir)
sens_sim_reader = pySWATPlus.TxtinoutReader(tio_dir=sens_sim_dir)

# Disable CSV print and set monthly output only (faster than daily)
sens_sim_reader.disable_csv_print()
sens_sim_reader.enable_object_in_print_prt(
    obj=None, daily=False, monthly=True, yearly=True, avann=True
)

# Trial run to verify the configuration produces the expected output files
sens_sim_reader.run_swat(
    begin_date = model_config['periods']['sensitivity_start'],
    end_date   = model_config['periods']['sensitivity_end'],
    warmup     = model_config['periods']['warmup_years'],
    print_prt_control={'channel_sd': {}}   # enable all time series for channel_sd
)

print("Sensitivity configuration complete. Trial run finished.")
print(f"Sensitivity sim dir    : {sens_sim_dir}")
print(f"Sensitivity results dir: {sens_results_dir}")

## Cell 6 — Run Sobol sensitivity analysis (63 parameters)

Sobol variance-based sensitivity indices are computed via `pySWATPlus.SensitivityAnalyzer`.  
The `seed` parameter is fixed to `SOBOL_SEED = 42` so sample generation is identical on any machine.

**S1** = first-order index (parameter alone)  
**ST** = total-order index (parameter + all interactions)

Parameters with high ST drive calibration effort; low-ST parameters are informative but less critical.

With `sample_number=4` and 63 parameters, total SWAT+ runs ≈ (2×63 + 2) × 4 = 512 runs.

In [ ]:
# ==========================================
# CELL 6: SOBOL SENSITIVITY ANALYSIS
# seed=SOBOL_SEED ensures identical sample generation on any machine
# ==========================================

# 63 parameters covering snow, ET, runoff, lateral flow,
# aquifer, channel, sediment, nitrogen, and phosphorus processes
# Bounds derived from physical constraints and literature
# (Arnold et al. 2012; Neitsch et al. 2011; Abbaspour et al. 2018)
parameters = [
    # ===== Snow =====
    {'name': 'snomelt_tmp', 'change_type': 'absval', 'lower_bound': 2.15,   'upper_bound': 2.20},
    {'name': 'snofall_tmp', 'change_type': 'absval', 'lower_bound': 1.60,   'upper_bound': 3.00},
    # ===== Evapotranspiration =====
    {'name': 'esco',        'change_type': 'absval', 'lower_bound': 0.25,   'upper_bound': 0.30},
    {'name': 'epco',        'change_type': 'absval', 'lower_bound': 0.45,   'upper_bound': 0.55},
    {'name': 'awc',         'change_type': 'pctchg', 'lower_bound': -0.40,  'upper_bound': -0.25},
    {'name': 'canmx',       'change_type': 'pctchg', 'lower_bound': -0.60,  'upper_bound': -0.40},
    # ===== Surface Runoff =====
    {'name': 'cn2',         'change_type': 'pctchg', 'lower_bound': 0.60,   'upper_bound': 0.75},
    {'name': 'cn3_swf',     'change_type': 'absval', 'lower_bound': 0.21,   'upper_bound': 0.27},
    {'name': 'ovn',         'change_type': 'pctchg', 'lower_bound': -0.85,  'upper_bound': -0.80},
    {'name': 'surlag',      'change_type': 'absval', 'lower_bound': 10.0,   'upper_bound': 13.5},
    # ===== Lateral Flow =====
    {'name': 'lat_ttime',   'change_type': 'pctchg', 'lower_bound': 4.00,   'upper_bound': 4.50},
    {'name': 'lat_len',     'change_type': 'abschg', 'lower_bound': 40.0,   'upper_bound': 48.0},
    {'name': 'latq_co',     'change_type': 'absval', 'lower_bound': 0.40,   'upper_bound': 0.46},
    {'name': 'bd',          'change_type': 'pctchg', 'lower_bound': 0.53,   'upper_bound': 0.70},
    {'name': 'k',           'change_type': 'pctchg', 'lower_bound': -1.00,  'upper_bound': -0.94},
    # ===== Aquifer =====
    {'name': 'perco',       'change_type': 'absval', 'lower_bound': 0.01,   'upper_bound': 0.10},
    {'name': 'flo_min',     'change_type': 'abschg', 'lower_bound': 30.0,   'upper_bound': 37.0},
    {'name': 'revap_co',    'change_type': 'absval', 'lower_bound': 0.01,   'upper_bound': 0.07},
    {'name': 'revap_min',   'change_type': 'abschg', 'lower_bound': 2.50,   'upper_bound': 6.50},
    {'name': 'alpha',       'change_type': 'absval', 'lower_bound': 0.45,   'upper_bound': 0.60},
    {'name': 'sp_yld',      'change_type': 'absval', 'lower_bound': 0.13,   'upper_bound': 0.19},
    {'name': 'bf_max',      'change_type': 'absval', 'lower_bound': 0.70,   'upper_bound': 0.90},
    {'name': 'deep_seep',   'change_type': 'absval', 'lower_bound': 0.30,   'upper_bound': 0.34},
    # ===== Channel =====
    {'name': 'chn',         'change_type': 'absval', 'lower_bound': 0.20,   'upper_bound': 0.22},
    {'name': 'evol',        'change_type': 'absval', 'lower_bound': 2500.0, 'upper_bound': 3000.0},
    {'name': 'pvol',        'change_type': 'absval', 'lower_bound': 40.0,   'upper_bound': 60.0},
    # ===== Sediment =====
    {'name': 'cov',         'change_type': 'absval', 'lower_bound': 0.30,   'upper_bound': 0.50},
    {'name': 'ch_clay',     'change_type': 'absval', 'lower_bound': 85.0,   'upper_bound': 100.0},
    {'name': 'chs',         'change_type': 'pctchg', 'lower_bound': -1.00,  'upper_bound': -0.50},
    {'name': 'cherod',      'change_type': 'absval', 'lower_bound': -0.06,  'upper_bound': 0.06},
    {'name': 'prf',         'change_type': 'absval', 'lower_bound': 0.00,   'upper_bound': 2.00},
    {'name': 'lat_sed',     'change_type': 'absval', 'lower_bound': 2000.0, 'upper_bound': 4000.0},
    {'name': 'usle_p',      'change_type': 'pctchg', 'lower_bound': -1.00,  'upper_bound': 1.00},
    {'name': 'adj_pkr',     'change_type': 'absval', 'lower_bound': 1.50,   'upper_bound': 2.00},
    # ===== Nitrogen =====
    {'name': 'n_updis',     'change_type': 'absval', 'lower_bound': 40.0,   'upper_bound': 100.0},
    {'name': 'nperco',      'change_type': 'pctchg', 'lower_bound': -1.00,  'upper_bound': 1.00},
    {'name': 'sdnco',       'change_type': 'pctchg', 'lower_bound': -1.00,  'upper_bound': 1.00},
    {'name': 'cmn',         'change_type': 'pctchg', 'lower_bound': -1.00,  'upper_bound': 1.00},
    {'name': 'rsdco',       'change_type': 'absval', 'lower_bound': 0.02,   'upper_bound': 1.00},
    {'name': 'hlife_n',     'change_type': 'absval', 'lower_bound': 170.0,  'upper_bound': 178.0},
    {'name': 'no3_init',    'change_type': 'absval', 'lower_bound': 1.80,   'upper_bound': 1.88},
    {'name': 'lat_orgn',    'change_type': 'absval', 'lower_bound': 3.40,   'upper_bound': 4.80},
    {'name': 'rs4',         'change_type': 'absval', 'lower_bound': 0.001,  'upper_bound': 0.0016},
    {'name': 'bc3',         'change_type': 'absval', 'lower_bound': 0.10,   'upper_bound': 0.45},
    {'name': 'bc1',         'change_type': 'absval', 'lower_bound': 0.60,   'upper_bound': 0.95},
    {'name': 'bc2',         'change_type': 'absval', 'lower_bound': 1.90,   'upper_bound': 1.98},
    {'name': 'rs3',         'change_type': 'absval', 'lower_bound': 0.01,   'upper_bound': 0.04},
    {'name': 'erorgn',      'change_type': 'absval', 'lower_bound': 0.80,   'upper_bound': 0.98},
    {'name': 'cdn',         'change_type': 'absval', 'lower_bound': 2.55,   'upper_bound': 2.75},
    {'name': 'orgn',        'change_type': 'absval', 'lower_bound': 4.80,   'upper_bound': 4.98},
    {'name': 'nh3',         'change_type': 'absval', 'lower_bound': 0.82,   'upper_bound': 0.98},
    {'name': 'no2',         'change_type': 'absval', 'lower_bound': 0.15,   'upper_bound': 0.25},
    {'name': 'nsetlr1',     'change_type': 'absval', 'lower_bound': 0.30,   'upper_bound': 0.355},
    {'name': 'nsetlr2',     'change_type': 'absval', 'lower_bound': 2.00,   'upper_bound': 4.00},
    # ===== Phosphorus =====
    {'name': 'p_updis',     'change_type': 'absval', 'lower_bound': 85.0,   'upper_bound': 100.0},
    {'name': 'pperco',      'change_type': 'absval', 'lower_bound': 5.00,   'upper_bound': 10.22},
    {'name': 'phoskd',      'change_type': 'absval', 'lower_bound': 190.0,  'upper_bound': 197.0},
    {'name': 'psp',         'change_type': 'absval', 'lower_bound': 0.0182, 'upper_bound': 0.0185},
    {'name': 'erorgp',      'change_type': 'absval', 'lower_bound': 2.80,   'upper_bound': 3.04},
    {'name': 'usle_k',      'change_type': 'absval', 'lower_bound': 0.54,   'upper_bound': 0.645},
    {'name': 'lat_orgp',    'change_type': 'absval', 'lower_bound': 1.00,   'upper_bound': 1.50},
    {'name': 'rs5',         'change_type': 'absval', 'lower_bound': 0.06,   'upper_bound': 0.089},
    {'name': 'bc4',         'change_type': 'absval', 'lower_bound': 0.20,   'upper_bound': 0.38},
]

print(f"Total sensitivity parameters: {len(parameters)}")

# Data extraction config — monthly channel output at Oulanka outlet
extract_data_sens = {
    'channel_sd_mon.txt': {
        'has_units': True,
        'ref_day'  : 1,
        'apply_filter': {'name': [CHANNEL_NAME]}
    }
}

observe_data_sens = {
    'channel_sd_mon.txt': {
        'obs_file'   : OBSERVED_FILE,
        'date_format': '%d-%m-%Y'
    }
}

metric_config_sens = {
    'channel_sd_mon.txt': {
        'sim_col'  : 'flo_out',
        'obs_col'  : '7300100_q',
        'indicator': 'NSE'
    }
}

# Run sensitivity analysis
# seed=SOBOL_SEED fixes the Sobol sample generation for reproducibility
if __name__ == '__main__':
    print("Starting Sobol sensitivity analysis...")
    print(f"Parameters: {len(parameters)} | sample_number: 4 | seed: {SOBOL_SEED}")
    print(f"Total SWAT+ runs: (2×{len(parameters)} + 2) × 4 = {(2*len(parameters)+2)*4}")

    sensitivity_output = pySWATPlus.SensitivityAnalyzer().simulation_and_indices(
        parameters   = parameters,
        sample_number= 4,              # increase for more reliable indices
        sensim_dir   = sens_results_dir,
        txtinout_dir = sens_sim_dir,
        extract_data = extract_data_sens,
        observe_data = observe_data_sens,
        metric_config= metric_config_sens,
        seed         = SOBOL_SEED      # fixed seed for reproducibility
    )

    print("Sensitivity analysis complete.")

    # Save Sobol indices to CSV
    fname_key = list(sensitivity_output.keys())[0]
    res = sensitivity_output[fname_key]
    s1_vals = list(res['S1'].values()) if isinstance(res['S1'], dict) else list(res['S1'])
    st_vals = list(res['ST'].values()) if isinstance(res['ST'], dict) else list(res['ST'])

    sens_df = pd.DataFrame({
        'parameter': [p['name'] for p in parameters],
        'S1'       : s1_vals,
        'ST'       : st_vals
    })
    sens_csv = pred_dir / "sensitivity_results.csv"
    sens_df.to_csv(sens_csv, index=False)
    print(f"Sensitivity results saved: {sens_csv}")

## Cell 7 — Plot sensitivity results

Grouped bar chart of Sobol S1 and ST indices. Top 20 by ST are shown for clarity.

In [ ]:
# ==========================================
# CELL 7: SENSITIVITY RESULT PLOT
# ==========================================

# Load from CSV so this cell can be re-run independently
sens_df = pd.read_csv(pred_dir / "sensitivity_results.csv")

# Clamp small negatives to zero (numerical noise in Sobol estimation)
sens_df['S1'] = sens_df['S1'].clip(lower=0)
sens_df['ST'] = sens_df['ST'].clip(lower=0)

top20 = sens_df.nlargest(20, 'ST').sort_values('ST', ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
x     = np.arange(len(top20))
width = 0.35

ax.barh(x - width/2, top20['S1'], width, label='First-order (S1)',
        color='#3498db', edgecolor='black', alpha=0.8)
ax.barh(x + width/2, top20['ST'], width, label='Total-order (ST)',
        color='#e67e22', edgecolor='black', alpha=0.8)

ax.set_yticks(x)
ax.set_yticklabels(top20['parameter'], fontsize=9)
ax.set_xlabel('Sobol Sensitivity Index', fontsize=11)
ax.set_title('Top 20 SWAT+ Parameters by Total-Order Sensitivity (ST)\nOulanka River Catchment', fontsize=12)
ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(pred_dir / "sensitivity_plot.png", dpi=150)
plt.show()
print("Sensitivity plot saved.")

## Cell 8 — Calibration (NSGA2, ~20–25 hours)

NSGA2 optimises the 63-parameter space to maximise NSE against monthly observed streamflow.  
The `seed` parameter is passed to pymoo to fix the evolutionary algorithm's initial population.

Calibration progress is logged to both the notebook and `calibration.log` in the calibration directory.

In [ ]:
# ==========================================
# CELL 8: NSGA2 CALIBRATION
# seed=NSGA2_SEED fixes initial population for reproducibility
# ==========================================

calibration_dir = os.path.join(BASE_DIR, "calibration", "NSGA2_Calibration")
if not os.path.exists(calibration_dir):
    os.makedirs(calibration_dir)

# Log calibration progress to file as well as the notebook
logging.basicConfig(
    filename=os.path.join(calibration_dir, 'calibration.log'),
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(message)s'
)

extract_data_cal = {
    'channel_sd_mon.txt': {
        'has_units'   : True,
        'ref_day'     : 1,
        'apply_filter': {'name': [CHANNEL_NAME]}
    }
}

observe_data_cal = {
    'channel_sd_mon.txt': {
        'obs_file'   : OBSERVED_FILE,
        'date_format': '%d-%m-%Y'
    }
}

objective_config_cal = {
    'channel_sd_mon.txt': {
        'sim_col'  : 'flo_out',
        'obs_col'  : '7300100_q',
        'indicator': 'NSE'
    }
}

if __name__ == '__main__':
    print(f"Starting NSGA2 calibration | n_gen={model_config['n_gen']} | pop_size={model_config['pop_size']} | seed={NSGA2_SEED}")
    print("Estimated runtime: 20–25 hours. Keep the VRE session active.")

    calibration = pySWATPlus.Calibration(
        parameters      = parameters,
        calsim_dir      = calibration_dir,
        txtinout_dir    = TXTINOUT_DIR,
        extract_data    = extract_data_cal,
        observe_data    = observe_data_cal,
        objective_config= objective_config_cal,
        algorithm       = model_config['algorithm'],
        n_gen           = model_config['n_gen'],
        pop_size        = model_config['pop_size'],
        seed            = NSGA2_SEED   # fixed seed for reproducibility
    )

    calibration_output = calibration.parameter_optimization()

    # Extract best solution from Pareto front
    if calibration_output['variables'].ndim > 1:
        optimized_values = calibration_output['variables'][0]
        best_nse         = float(calibration_output['objectives'][0][0])
    else:
        optimized_values = calibration_output['variables']
        best_nse         = float(calibration_output['objectives'][0])

    # Print readable summary
    print("\n" + "=" * 65)
    print("OPTIMIZED PARAMETER VALUES")
    print("=" * 65)
    print(f"Best NSE achieved : {best_nse:.4f}")
    print(f"Total runtime     : {calibration_output['time_sec']/3600:.2f} hours")
    print("-" * 65)
    print(f"{'No.':<5}{'Parameter':<15}{'Change Type':<12}{'Optimized Value':<15}")
    print("-" * 65)
    for i, p in enumerate(parameters):
        print(f"{i+1:<5}{p['name']:<15}{p['change_type']:<12}{optimized_values[i]:<15.6f}")
    print("=" * 65)

    # Save optimized parameters to CSV
    results_df = pd.DataFrame({
        'parameter'      : [p['name'] for p in parameters],
        'change_type'    : [p['change_type'] for p in parameters],
        'lower_bound'    : [p['lower_bound'] for p in parameters],
        'upper_bound'    : [p['upper_bound'] for p in parameters],
        'optimized_value': optimized_values
    })
    cal_csv = model_dir / "optimized_parameters.csv"
    results_df.to_csv(cal_csv, index=False)
    print(f"\nOptimized parameters saved: {cal_csv}")

## Cell 9 — Calibrated validation run

Run SWAT+ with the optimized parameters over the full historical period.  
Performance metrics are computed for both the calibration and validation periods.

In [ ]:
# ==========================================
# CELL 9: CALIBRATED VALIDATION RUN
# ==========================================

cal_sim_dir = os.path.join(BASE_DIR, "model_run_calibrated")
if not os.path.exists(cal_sim_dir):
    os.makedirs(cal_sim_dir)

cal_base_reader = pySWATPlus.TxtinoutReader(tio_dir=TXTINOUT_DIR)
cal_base_reader.copy_required_files(sim_dir=cal_sim_dir)
cal_sim_reader = pySWATPlus.TxtinoutReader(tio_dir=cal_sim_dir)

cal_sim_reader.enable_object_in_print_prt(
    obj='channel_sd', daily=True, monthly=True, yearly=True, avann=True
)

# Load the optimized parameters from disk
# (can be run independently of Cell 8 if CSV already exists)
opt_df = pd.read_csv(model_dir / "optimized_parameters.csv")
calibrated_params = [
    {'name': row['parameter'], 'change_type': row['change_type'], 'value': row['optimized_value']}
    for _, row in opt_df.iterrows()
]

print(f"Running calibrated SWAT+ with {len(calibrated_params)} optimized parameters...")
t0 = time.time()

cal_sim_reader.run_swat(
    begin_date = model_config['periods']['simulation_start'],
    end_date   = model_config['periods']['simulation_end'],
    warmup     = model_config['periods']['warmup_years'],
    parameters = calibrated_params
)

elapsed = time.time() - t0
mins, secs = divmod(elapsed, 60)
print(f"Calibrated run complete. Runtime: {int(mins)} min {secs:.1f} s")

## Cell 10 — Compute and save performance metrics

NSE, KGE, R², and PBIAS are computed for the calibration and validation periods separately.
Results are saved to CSV for use in SE4.

In [ ]:
# ==========================================
# CELL 10: PERFORMANCE METRICS
# ==========================================

# --- Metric functions ---
def nse(obs, sim):
    """Nash-Sutcliffe Efficiency. Optimal = 1."""
    return 1 - np.sum((obs - sim)**2) / np.sum((obs - np.mean(obs))**2)

def kge(obs, sim):
    """Kling-Gupta Efficiency. Optimal = 1."""
    r     = np.corrcoef(obs, sim)[0, 1]
    beta  = np.mean(sim) / np.mean(obs)
    gamma = (np.std(sim) / np.mean(sim)) / (np.std(obs) / np.mean(obs))
    return 1 - np.sqrt((r-1)**2 + (beta-1)**2 + (gamma-1)**2)

def r2(obs, sim):
    """Coefficient of determination. Optimal = 1."""
    return np.corrcoef(obs, sim)[0, 1]**2

def pbias(obs, sim):
    """Percent Bias. Optimal = 0. Positive = underestimate."""
    return 100 * np.sum(obs - sim) / np.sum(obs)


# --- Load calibrated simulation output ---
sim_output = os.path.join(cal_sim_dir, "channel_sd_day.txt")
df_sim_raw = pd.read_csv(sim_output, sep=r'\s+', skiprows=[0, 2])
df_sim_raw['date'] = pd.to_datetime(
    df_sim_raw[['yr', 'mon', 'day']].rename(columns={'yr': 'year', 'mon': 'month'})
)
df_sim = df_sim_raw[df_sim_raw['unit'] == TARGET_UNIT][['date', 'flo_out']].copy()

# --- Load observed flow ---
df_obs_flow = pd.read_csv(OBSERVED_FILE)
df_obs_flow['date'] = pd.to_datetime(df_obs_flow['date'], dayfirst=True)
df_obs_flow.columns = ['date', 'obs_q']

# --- Merge ---
merged = pd.merge(df_sim, df_obs_flow, on='date').dropna()

# Save merged predictions to CSV for SE4
merged.to_csv(pred_dir / "predictions_calval.csv", index=False)

# --- Compute metrics per period ---
periods = {
    'calibration': (model_config['periods']['calibration_start'][:7], model_config['periods']['calibration_end'][:7]),
    'validation' : (model_config['periods']['validation_start'][:7],  model_config['periods']['validation_end'][:7])
}

metrics_rows = {}
for period_name, (start, end) in periods.items():
    sub = merged[(merged['date'] >= start) & (merged['date'] <= end)]
    if sub.empty:
        print(f"Warning: no data for {period_name} period ({start} to {end})")
        continue
    obs_v = sub['obs_q'].values
    sim_v = sub['flo_out'].values
    metrics_rows[period_name] = {
        'NSE'  : nse(obs_v, sim_v),
        'KGE'  : kge(obs_v, sim_v),
        'R2'   : r2(obs_v, sim_v),
        'PBIAS': pbias(obs_v, sim_v)
    }

metrics_df = pd.DataFrame(metrics_rows).T
print(metrics_df.round(3))

metrics_df.to_csv(pred_dir / "evaluation_metrics.csv")
print(f"\nMetrics saved: {pred_dir / 'evaluation_metrics.csv'}")

# --- Hydrograph plot ---
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(merged['date'], merged['obs_q'],   color='#2166ac', lw=1.2, label='Observed',  zorder=3)
ax.plot(merged['date'], merged['flo_out'], color='#d6604d', lw=1.0, label='Simulated', zorder=2)
ax.axvspan(pd.Timestamp(model_config['periods']['calibration_start']),
           pd.Timestamp(model_config['periods']['calibration_end']),
           alpha=0.08, color='green',  label='Calibration')
ax.axvspan(pd.Timestamp(model_config['periods']['validation_start']),
           pd.Timestamp(model_config['periods']['validation_end']),
           alpha=0.08, color='orange', label='Validation')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.set_xlabel('Date')
ax.set_ylabel('Streamflow (m³/s)')
nse_c = metrics_df.loc['calibration', 'NSE'] if 'calibration' in metrics_df.index else float('nan')
nse_v = metrics_df.loc['validation',  'NSE'] if 'validation'  in metrics_df.index else float('nan')
ax.set_title(f"Oulanka River — Observed vs Simulated | Calib NSE={nse_c:.3f}  Valid NSE={nse_v:.3f}")
ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(pred_dir / "hydrograph_calval.png", dpi=150)
plt.show()
print("Hydrograph saved.")

## Summary

SE3a is complete. All outputs are written to `oulanka_swatplus_processeddata/`.  
**Run SE3b daily** to generate near-real-time predictions using the calibrated parameters.

---
## Need help?
https://github.com/orgs/DigitalWaters-fi/discussions — Tag **#modelling**